# Qorx Zero — public memory validation

This notebook independently validates the standalone Hackathon Edition with synthetic data. It contains no private Qorx implementation, datasets, or credentials.

In [ ]:
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from math import exp
import re

STOP = {'a','an','and','are','as','at','be','by','for','from','how','i','in','is','it','must','of','on','or','should','that','the','this','to','we','what','when','where','which','who','will','with'}

@dataclass(frozen=True)
class Memory:
    id: str
    text: str
    tags: tuple[str, ...]
    importance: int
    last_used_at: datetime
    expires_at: datetime | None = None

def tokens(value: str) -> set[str]:
    return {token for token in re.sub(r'[^a-z0-9_-]+', ' ', value.lower()).split() if len(token) > 1 and token not in STOP}

def rank(query: str, memories: list[Memory], now: datetime) -> list[tuple[float, Memory]]:
    query_terms = tokens(query)
    ranked = []
    for memory in memories:
        if memory.expires_at is not None and memory.expires_at <= now:
            continue
        matches = query_terms & tokens(memory.text + ' ' + ' '.join(memory.tags))
        if not matches or not query_terms:
            continue
        relevance = len(matches) / len(query_terms)
        age_days = max(0.0, (now - memory.last_used_at).total_seconds() / 86400)
        recency = exp(-age_days / 30)
        importance = min(5, max(1, memory.importance)) / 5
        ranked.append((relevance * 0.65 + importance * 0.20 + recency * 0.15, memory))
    return sorted(ranked, key=lambda item: (item[0], item[1].importance, item[1].last_used_at), reverse=True)

def proof_frame(query: str, memories: list[Memory], now: datetime, max_items: int = 5, max_chars: int = 1600):
    selected, lines = [], []
    for score, memory in rank(query, memories, now):
        if len(selected) >= max_items:
            break
        line = f'- [{memory.id}; importance={memory.importance}] {memory.text}'
        candidate = '\n'.join(lines + [line])
        if len(candidate) > max_chars:
            remaining = max_chars - len('\n'.join(lines)) - (1 if lines else 0)
            if remaining > 48:
                lines.append(line[:remaining - 1] + '…')
                selected.append(memory)
            break
        lines.append(line)
        selected.append(memory)
    return '\n'.join(lines), selected


In [ ]:
NOW = datetime(2026, 7, 19, 12, tzinfo=timezone.utc)
memories = [
    Memory('launch-a1', 'Ship the memory demo and forgetting flow before adding analytics.', ('launch','priority','analytics'), 5, NOW),
    Memory('design-b2', 'Use high-contrast colors and keyboard-accessible controls.', ('design','accessibility'), 4, NOW - timedelta(days=1)),
    Memory('expired-c3', 'Old analytics plan that must be forgotten.', ('analytics',), 5, NOW - timedelta(days=20), NOW - timedelta(days=1)),
]

frame, selected = proof_frame('What must we ship before adding analytics?', memories, NOW)
print(frame)
print(f'\nSelected: {len(selected)} of {len(memories)} records; frame: {len(frame)} characters')


In [ ]:
def run_checks():
    checks = []
    frame, selected = proof_frame('ship analytics', memories, NOW)
    checks.append(('relevant memory ranks first', selected[0].id == 'launch-a1'))
    checks.append(('expired memory is absent', all(item.id != 'expired-c3' for item in selected)))
    unrelated_frame, unrelated = proof_frame('database password', memories, NOW)
    checks.append(('unrelated memory is excluded', unrelated_frame == '' and unrelated == []))
    after_forget = [item for item in memories if item.id != 'launch-a1']
    _, recalled = proof_frame('ship analytics', after_forget, NOW)
    checks.append(('explicit forgetting survives recall', all(item.id != 'launch-a1' for item in recalled)))
    long_memories = [Memory(f'cap-{i}', 'Launch ' + ('x' * 180), ('launch',), 5, NOW - timedelta(seconds=i)) for i in range(10)]
    capped, capped_selected = proof_frame('launch', long_memories, NOW, max_items=3, max_chars=240)
    checks.append(('proof character cap holds', len(capped) <= 240))
    checks.append(('proof record cap holds', len(capped_selected) <= 3))
    for name, passed in checks:
        print(('PASS' if passed else 'FAIL') + ' — ' + name)
    assert all(passed for _, passed in checks)
    return len(checks)

count = run_checks()
print(f'\nQorx Zero validation complete: {count}/{count} checks passed.')


## Interpretation

The notebook checks the public algorithm only. Passing results support four narrow claims: relevant records can be recalled, expired and explicitly forgotten records stay out, unrelated records are excluded, and the proof frame remains bounded.